In [1]:
import numpy as np
import pandas as pd
import pickle
import random

In [17]:
model = ''
dataset ='' 
model_dataset = pd.read_csv(f'test_result/generate/{model}_sample_{dataset}_generate_fake_False.csv')


In [3]:
def process_output(output):
    split_output = output.split('###Conditions:', 1)
    if len(split_output) > 1:
        return split_output[1].strip()
    return output.strip()

def calculate_output_length(df):
    df['output'] = df['output'].apply(process_output)
    df['output_length'] = df['output'].apply(lambda x: len(x))
    return df

In [6]:
from tqdm import tqdm

def eval_df(df, n=3): #Partial Match

    df['em_n'] = 0
    df['recall_em'] = 0
    is_in = {}
    for i in (range(len(df))):
        condition_list = df['condition'][i].replace("[", "").replace("]", "").replace("'", "").split(", ")
        target_list = list(set(condition_list))
        split_by_space = [cond.split() for cond in target_list]
        flattened_list = [word for condition in split_by_space for word in condition]
        flattened_list = list(set(flattened_list))
        flattened_list = [word for word in flattened_list if len(word) > n]
        
        opt_em_n = 0
        
        for j in range(len(flattened_list)):
            target = flattened_list[j].lower()
            
            if target in df.output[i].lower():
                opt_em_n += 1
                is_in[target] = is_in.get(target, 0) + 1
        
        df.loc[i, 'em_n'] = int(opt_em_n)
        df.loc[i, 'recall_em'] = float(opt_em_n) / len(flattened_list)

        is_in_df = pd.DataFrame(list(is_in.items()), columns=['condition', 'count'])
        is_in_df = is_in_df.sort_values(by='count', ascending=False)

    return df, is_in_df


In [7]:
def exact_match_df(df):

    df['em_n'] = 0
    df['recall_em'] = 0
    is_in = {}
    for i in (range(len(df))):
        condition_list = df['condition'][i].replace("[", "").replace("]", "").replace("'", "").split(", ")
        target_list = []
        for condition in set(condition_list):
            if 'NEC/NOS' in condition:
                condition = condition.replace('NEC/NOS', '').strip()
            elif 'NOS' in condition:
                condition = condition.replace('NOS', '').strip()
            elif 'NEC' in condition:
                condition = condition.replace('NEC', '').strip()
            if condition: 
                target_list.append(condition)
        
        opt_em_n = 0
        for j in range(len(target_list)):
            target = target_list[j].lower()
            
            if target in df.output[i].lower():
                opt_em_n += 1
                
                is_in[target] = is_in.get(target, 0) + 1

        df.loc[i, 'em_n'] = int(opt_em_n)
        df.loc[i, 'recall_em'] = float(opt_em_n) / len(target_list)

        is_in_df = pd.DataFrame(list(is_in.items()), columns=['condition', 'count'])
        is_in_df = is_in_df.sort_values(by='count', ascending=False)

    return df, is_in_df

In [ ]:
model_dataset_eval, pred_model_dataset = eval_df(model_dataset)
model_dataset_exact_eval, pred_model_dataset_exact = exact_match_df(model_dataset)


In [ ]:
print("### Partial word match ###")
print('model_dataset_eval: ', np.average(model_dataset_eval.recall_em))

print("### Exact match ###")
print('model_dataset_eval: ', np.average(model_dataset_exact_eval.recall_em))
